In [ ]:
import os, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import glob
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader

## Load tokenizer and model

In [ ]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B", "Qwen/Qwen3-8B"]

def load(MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    model.eval()
    model = torch.compile(model, mode="reduce-overhead")
    print(f"Model loaded on : {next(model.parameters()).device}")
    return tokenizer, model

## ELGOLD Dataset

Used **Elgold: gold standard, multi-genre dataset for named entity recognition and linking** for this benchmark.

Dataset link: [Elgold dataset](https://mostwiedzy.pl/en/open-research-data/elgold-gold-standard-multi-genre-dataset-for-named-entity-recognition-and-linking,202503241553453028951-0)

The marked entities are embedded directly in the textual files using the following syntax:

{{mention content|entity class|Wikipedia target}}

In [ ]:
DATA_DIR     = "../data/"
MAX_EXAMPLES = None
BATCH_SIZE = 64

In [ ]:
ANNOT_RE = re.compile(r"\{\{([^|{}]+)\|([^|{}]+)\|([^{}]+)\}\}")

def clean_text_and_extract_mentions(path):
    raw = Path(path).read_text(encoding="utf-8")
    examples = []

    def repl(match):
        mention = match.group(1).strip()
        ent_type = match.group(2).strip()
        target = match.group(3).strip()

        examples.append({
            "file": Path(path).name,
            "mention": mention,
            "entity_type": ent_type,
            "gold": target,
        })

        return mention

    clean_text = ANNOT_RE.sub(repl, raw)

    for ex in examples:
        ex["context"] = clean_text

    return examples

rows = []
for file in glob.glob(f"{DATA_DIR}/*.txt"):
    rows.extend(clean_text_and_extract_mentions(file))

df_gold = pd.DataFrame(rows).drop_duplicates()
if MAX_EXAMPLES is not None:
    df_gold = df_gold.head(MAX_EXAMPLES)

print(df_gold.head())
print("Examples number:", len(df_gold))

## Prompt zero-shot

In [ ]:
def build_prompt(text: str, mention: str, entity_type: str) -> str:
    return f"""You are an Entity Linking system. Your task is to identify the entity that a mention refers to in a text, and return its canonical name as it appears in a Knowledge Base (e.g. Wikidata / Wikipedia).

Text: "{text}"
Mention: "{mention}"
Entity type: {entity_type}

Instructions:
- Identify which real-world entity the mention "{mention}" refers to in this specific context.
- Return ONLY the canonical entity name, nothing else.
- Do not explain. Do not add punctuation. Just the name.

Canonical entity name:"""

In [ ]:
def generate_batch(prompts: list[str], tokenizer, model, max_new_tokens=20) -> list[str]:
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    results = []
    input_len = inputs["input_ids"].shape[1]
    for out in outputs:
        new_tokens = out[input_len:]
        results.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return results

In [ ]:
def parse_answer(raw: str) -> str:
    for line in raw.splitlines():
        line = line.strip()
        if line:
            return line
    return raw.strip()

In [ ]:
# First example
row = df_gold.iloc[0]
print("PROMPT :")
print("-" * 60)
print(build_prompt(row["context"], row["mention"], row["entity_type"]))
print("-" * 60)
print(f"Gold expected : {row['gold']}")

## Zero-shot Evaluation

In [ ]:
results = dict()

for MODEL in MODELS:
    print(f"\n{'='*50}\nModel : {MODEL}\n{'='*50}")

    tokenizer, model = load(MODEL)
    results[MODEL] = []
    
    rows_data = [
        {
            "file":        row["file"],
            "mention":     row["mention"],
            "entity_type": row["entity_type"],
            "gold":        row["gold"],
            "prompt":      build_prompt(row["context"], row["mention"], row["entity_type"]),
        }
        for _, row in df_gold.iterrows()
    ]

    for i in tqdm(range(0, len(rows_data), BATCH_SIZE), desc=f"Zero-shot {MODEL}"):
        batch = rows_data[i:i + BATCH_SIZE]
        batch_prompts  = [item["prompt"] for item in batch]
        batch_outputs  = generate_batch(batch_prompts, tokenizer, model)
        for item, raw_output in zip(batch, batch_outputs):
            pred = parse_answer(raw_output)
            results[MODEL].append({
                "file":        item["file"],
                "mention":     item["mention"],
                "entity_type": item["entity_type"],
                "gold":        item["gold"],
                "predicted":   pred,
                "correct":     pred.lower().strip() == item["gold"].lower().strip(),
                "raw_output":  raw_output,
            })

    df_results = pd.DataFrame(results[MODEL])
    total = len(df_results)
    acc = df_results["correct"].mean()

    print("Zero-shot ELGOLD results:\n")
    print(f"Model: {MODEL}")
    print(f"Tested examples: {total}")
    print(f"Accuracy: {acc:.2%}  ({df_results['correct'].sum()}/{total})")

    model_slug = MODEL.replace("/", "_")
    df_results.to_csv(f"zero_shot_elgold_{model_slug}.csv", index=False)

    wrong = df_results[df_results["correct"] == False]
    print(f"\n{len(wrong)} errors :")
    for _, r in wrong.head(10).iterrows():
        print(f"  '{r['mention']}' | gold={r['gold']} | pred={r['predicted']}")

    # Free GPU memory before the next model
    del model, tokenizer
    torch.cuda.empty_cache()

## Model comparison

In [ ]:
rows = []
for MODEL, res in results.items():
    df_r = pd.DataFrame(res)
    rows.append({
        "model":     MODEL,
        "accuracy":  df_r["correct"].mean(),
        "n_correct": int(df_r["correct"].sum()),
        "n_total":   len(df_r),
    })

df_compare = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
df_compare["accuracy"] = df_compare["accuracy"].map("{:.2%}".format)
print("ZERO-SHOT Model comparison")
print(df_compare.to_string(index=False))